# Task 1 · Step 1 — Standard DPO (one epoch)

Truncation policy report, SFT reference evaluation, one-epoch DPO, held-out evaluation (held-out DPO loss / preference accuracy on the filtered pairs; RM score, sampled KL, entropy and length on 200 held-out prompts).

**Research questions (manual)**
- Is stronger preference fitting accompanied by useful behaviour, or by disproportionate drift / a trivial length shift?

This notebook only runs the step's scripts and shows their saved outputs; interpretation is left to the report.

In [ ]:
import os, sys, glob
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "configs").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import Image, Markdown, display
pd.set_option("display.max_colwidth", 80)

def show_tables(*names):
    for n in names:
        p = ROOT / "report" / "tables" / f"{n}.csv"
        if p.exists():
            display(Markdown(f"**{n}**")); display(pd.read_csv(p))
        else:
            print(f"[missing] {p} - run the step, then `python -m common.visualize_all`")

def show_figures(*names):
    for n in names:
        hits = sorted(glob.glob(str(ROOT / "report" / "figures" / n)))
        if not hits:
            print(f"[missing] report/figures/{n}")
        for h in hits:
            display(Markdown(f"`{Path(h).name}`")); display(Image(filename=h, width=1100))
print("repo root:", ROOT)

## Run (optional; the Kaggle runner already executes these)

In [ ]:
RUN = False   # set True to (re)run this step here
if RUN:
    !python -m task1_dpo.preprocess --config configs/dpo.yaml
    !python -m task1_dpo.evaluate --config configs/dpo.yaml --adapter none --name sft_reference
    !python -m task1_dpo.train --config configs/dpo.yaml --run-name standard
    !python -m task1_dpo.evaluate --config configs/dpo.yaml --adapter outputs/task1_dpo/standard --name standard

## Tables

In [ ]:
import subprocess
subprocess.run([sys.executable, '-m', 'common.visualize_all'], capture_output=True)  # rebuild report/tables
show_tables('t1_truncation_policy', 't1_dpo_summary')

## Figures

In [ ]:
show_figures('task1_dpo_truncation_policy.png', 'task1_dpo_training_curves_standard.png', 'task1_dpo_implicit_reward_standard.png', 'task1_dpo_eval_sft_reference.png', 'task1_dpo_eval_standard.png')